# Document Augmentation

Enrich the index with generated questions that point back to the original chunk.

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
def make_questions(chunk, n=3):
    data = llm_json(f"Write {n} questions a user might ask that this passage answers. Reply with a JSON list of strings.\n\nPassage: {chunk}")
    return [q for q in (data or []) if isinstance(q, str)]

aug_texts, aug_meta = [], []
for i, chunk in enumerate(chunks):
    aug_texts.append(chunk); aug_meta.append({"chunk": i, "kind": "chunk"})
    for q in make_questions(chunk):
        aug_texts.append(q); aug_meta.append({"chunk": i, "kind": "question"})

aug_store = build_store(aug_texts, aug_meta)
print(f"{len(chunks)} chunks + {len(aug_texts) - len(chunks)} generated questions")

Unlike HyPE, the original chunk vectors stay in the index; questions are *extra* entry points. Deduplicate by source chunk at query time.

In [ ]:
def augmented_search(query, k=3):
    seen, out = set(), []
    for h in aug_store.search(query, k * 4):
        c = h["meta"]["chunk"]
        if c not in seen:
            seen.add(c)
            out.append({"idx": c, "text": chunks[c], "score": h["score"], "meta": h["meta"]})
    return out[:k]

query = "When should I use a smaller model in a RAG pipeline?"
hits = augmented_search(query)
for h in hits:
    print(h["meta"]["kind"], f"{h['score']:.3f}", h["text"][:100])
print(answer(query, [h["text"] for h in hits]))